<a href="https://colab.research.google.com/github/StrawEater/PracticasPDI3erBimestre/blob/main/Laboratorios/labo4/Laboratorio_4_visualizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# @title
from IPython.display import HTML, display

display(HTML("""
<style>
#lab-wrapper{
  background: #1e1e24;
  color: #f0f0f0;
  border-radius: 12px;
  padding: 16px 12px;
  box-shadow: 0 4px 16px rgba(0,0,0,0.5);
  border: 1px solid #33333f;
  max-width: 820px;
  margin: 10px auto;
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
  text-align: center;
  user-select: none;
}

#lab-wrapper h2{
  margin: 4px 0 12px 0;
  color: #ffffff;
  font-size: 20px;
  letter-spacing: 0.5px;
}

.canvas-wrap{
  display: inline-block;
  margin: 6px;
  background: #121216;
  padding: 8px;
  border-radius: 8px;
  border: 1px solid #2a2a35;
}

#lab-wrapper canvas{
  width: 350px;
  height: 350px;
  background: #000;
  border: 1px solid #444;
  border-radius: 4px;
  display: block;
  cursor: crosshair;
  touch-action: none;
}

.label-badge{
  display: inline-block;
  font-weight: 600;
  font-size: 13px;
  margin-bottom: 8px;
  padding: 4px 10px;
  background: rgba(255, 255, 255, 0.08);
  color: #e0e0e0;
  border-radius: 20px;
  border: 1px solid rgba(255, 255, 255, 0.15);
  letter-spacing: 0.3px;
}

.controls-bar{
  margin-top: 10px;
  display: flex;
  flex-wrap: wrap;
  justify-content: center;
  align-items: center;
  gap: 6px;
}

#lab-wrapper button, #lab-wrapper input[type="file"]{
  padding: 6px 12px;
  border-radius: 6px;
  border: 1px solid #4a4a5a;
  background: #2b2b36;
  color: #ffffff;
  font-size: 13px;
  cursor: pointer;
  transition: background 0.15s ease, border-color 0.15s ease;
}

#lab-wrapper button:hover:not(:disabled){
  background: #3b3b4a;
  border-color: #6a6a80;
}

#lab-wrapper button:disabled{
  opacity: 0.4;
  cursor: not-allowed;
}

.range-wrap{
  display: inline-flex;
  align-items: center;
  gap: 8px;
  font-size: 13px;
  color: #ddd;
  background: #2b2b36;
  padding: 4px 10px;
  border-radius: 6px;
  border: 1px solid #4a4a5a;
}

#lab-wrapper input[type="range"]{
  cursor: pointer;
  accent-color: #4a90e2;
}

#lab-wrapper input[type="file"]{
  max-width: 250px;
}
</style>

<div id="lab-wrapper">
  <h2>🔬 Laboratorio Interactivo de Fourier</h2>

  <div style="margin-bottom: 10px;">
    <input type="file" id="file" accept="image/*">
  </div>

  <div class="canvas-wrap">
    <div class="label-badge">🎨 Dominio Espacial (Imagen)</div>
    <canvas id="spatial" width="256" height="256"></canvas>
  </div>

  <div class="canvas-wrap">
    <div class="label-badge">✨ Dominio de Frecuencia (Fourier)</div>
    <canvas id="fourier" width="256" height="256"></canvas>
  </div>

  <div class="controls-bar">
    <button id="pencil">✏️ Dibujar</button>
    <button id="eraser">🧹 Borrar</button>
    <button id="undo" disabled>↶ Deshacer (0)</button>
    <button id="clear">🗑️ Limpiar Todo</button>
    <button id="save">💾 Guardar Imagen</button>

    <div class="range-wrap">
      <span>Grosor</span>
      <input type="range" id="size" min="1" max="18" value="3">
    </div>
  </div>
</div>

<script>
(()=>{
const N = 256;
const MAX_HISTORY = 25;
const F = document.getElementById("fourier");
const S = document.getElementById("spatial");
const undoBtn = document.getElementById("undo");
const fc = F.getContext("2d");
const sc = S.getContext("2d");

let re = new Float64Array(N * N);
let im = new Float64Array(N * N);
let spatial = new Float64Array(N * N);

const historyStack = [];
let brush = 3, erase = false, drawing = false, activeCanvas = null;
let lastX = -1, lastY = -1;

function fft(a, b, inv) {
  for (let i = 1, j = 0; i < N; i++) {
    let bit = N >> 1;
    for (; j & bit; bit >>= 1) j ^= bit;
    j ^= bit;
    if (i < j) {
      [a[i], a[j]] = [a[j], a[i]];
      [b[i], b[j]] = [b[j], b[i]];
    }
  }
  for (let l = 2; l <= N; l <<= 1) {
    let A = (inv ? 1 : -1) * 2 * Math.PI / l;
    let wr = Math.cos(A), wi = Math.sin(A);
    for (let i = 0; i < N; i += l) {
      let cr = 1, ci = 0, h = l >> 1;
      for (let j = 0; j < h; j++) {
        let x = i + j, y = x + h;
        let tr = a[y] * cr - b[y] * ci;
        let ti = a[y] * ci + b[y] * cr;
        a[y] = a[x] - tr;
        b[y] = b[x] - ti;
        a[x] += tr;
        b[x] += ti;
        [cr, ci] = [cr * wr - ci * wi, cr * wi + ci * wr];
      }
    }
  }
  if (inv) for (let i = 0; i < N; i++) a[i] /= N, b[i] /= N;
}

function fft2(a, b, inv) {
  let r = new Float64Array(N), q = new Float64Array(N);
  for (let y = 0; y < N; y++) {
    for (let x = 0; x < N; x++) r[x] = a[y * N + x], q[x] = b[y * N + x];
    fft(r, q, inv);
    for (let x = 0; x < N; x++) a[y * N + x] = r[x], b[y * N + x] = q[x];
  }
  for (let x = 0; x < N; x++) {
    for (let y = 0; y < N; y++) r[y] = a[y * N + x], q[y] = b[y * N + x];
    fft(r, q, inv);
    for (let y = 0; y < N; y++) a[y * N + x] = r[y], b[y * N + x] = q[y];
  }
}

function renderFourier() {
  let z = fc.createImageData(N, N), maxVal = 0;
  for (let y = 0; y < N; y++) {
    for (let x = 0; x < N; x++) {
      let X = (x + N / 2) & 255, Y = (y + N / 2) & 255, i = Y * N + X;
      let mag = Math.log1p(Math.hypot(re[i], im[i]));
      if (mag > maxVal) maxVal = mag;
    }
  }
  for (let y = 0; y < N; y++) {
    for (let x = 0; x < N; x++) {
      let X = (x + N / 2) & 255, Y = (y + N / 2) & 255, i = Y * N + X;
      let v = maxVal > 0 ? (Math.log1p(Math.hypot(re[i], im[i])) * 255 / maxVal) : 0;
      let p = (y * N + x) * 4;
      z.data[p] = z.data[p + 1] = z.data[p + 2] = v;
      z.data[p + 3] = 255;
    }
  }
  fc.putImageData(z, 0, 0);
}

function renderSpatial() {
  let z = sc.createImageData(N, N);
  for (let i = 0; i < N * N; i++) {
    let v = Math.min(255, Math.max(0, spatial[i]));
    let p = i * 4;
    z.data[p] = z.data[p + 1] = z.data[p + 2] = v;
    z.data[p + 3] = 255;
  }
  sc.putImageData(z, 0, 0);
}

function updateSpatialFromFourier() {
  let a = new Float64Array(re), b = new Float64Array(im);
  fft2(a, b, true);
  for (let i = 0; i < N * N; i++) spatial[i] = a[i];
  renderSpatial();
}

function updateFourierFromSpatial() {
  for (let i = 0; i < N * N; i++) {
    re[i] = spatial[i];
    im[i] = 0;
  }
  fft2(re, im, false);
  renderFourier();
}

function updateUndoButton() {
  undoBtn.disabled = historyStack.length === 0;
  undoBtn.innerText = `↶ Deshacer (${historyStack.length})`;
}

function saveState() {
  if (historyStack.length >= MAX_HISTORY) {
    historyStack.shift();
  }
  historyStack.push({
    spatial: new Float64Array(spatial),
    re: new Float64Array(re),
    im: new Float64Array(im)
  });
  updateUndoButton();
}

function applyBrushAt(cx, cy, canvas) {
  for (let dy = -brush; dy <= brush; dy++) {
    for (let dx = -brush; dx <= brush; dx++) {
      if (dx * dx + dy * dy <= brush * brush) {
        let x = cx + dx, y = cy + dy;
        if (x >= 0 && x < N && y >= 0 && y < N) {
          if (canvas === S) {
            spatial[y * N + x] = erase ? 0 : 255;
          } else {
            let X = (x + N / 2) & 255, Y = (y + N / 2) & 255;
            let i1 = Y * N + X;
            let symX = (N - X) & 255, symY = (N - Y) & 255;
            let i2 = symY * N + symX;

            if (erase) {
              re[i1] = im[i1] = 0;
              re[i2] = im[i2] = 0;
            } else {
              let boost = 3000;
              re[i1] += boost;
              re[i2] += boost;
            }
          }
        }
      }
    }
  }
}

function interpolateAndDraw(x1, y1, x2, y2, canvas) {
  let dist = Math.hypot(x2 - x1, y2 - y1);
  let steps = Math.max(1, Math.ceil(dist / 2));
  for (let s = 0; s <= steps; s++) {
    let t = s / steps;
    let cx = Math.round(x1 + (x2 - x1) * t);
    let cy = Math.round(y1 + (y2 - y1) * t);
    applyBrushAt(cx, cy, canvas);
  }
}

function handlePointer(e, isDown = false) {
  if (!drawing && !isDown) return;
  let rect = activeCanvas.getBoundingClientRect();
  let x = Math.floor((e.clientX - rect.left) / rect.width * N);
  let y = Math.floor((e.clientY - rect.top) / rect.height * N);

  if (isDown) {
    lastX = x; lastY = y;
    applyBrushAt(x, y, activeCanvas);
  } else {
    interpolateAndDraw(lastX, lastY, x, y, activeCanvas);
    lastX = x; lastY = y;
  }

  if (activeCanvas === S) {
    renderSpatial();
    updateFourierFromSpatial();
  } else {
    renderFourier();
    updateSpatialFromFourier();
  }
}

function startStroke(e, canvas) {
  saveState();
  drawing = true;
  activeCanvas = canvas;
  handlePointer(e, true);
}

S.onpointerdown = e => startStroke(e, S);
F.onpointerdown = e => startStroke(e, F);

window.addEventListener("pointermove", e => { if (drawing) handlePointer(e, false); });
window.addEventListener("pointerup", () => { drawing = false; activeCanvas = null; });

function loadImage(file) {
  if (!file) return;
  let img = new Image(), u = URL.createObjectURL(file);
  img.onload = () => {
    let c = document.createElement("canvas"), x = c.getContext("2d");
    c.width = c.height = N;
    x.drawImage(img, 0, 0, N, N);
    let d = x.getImageData(0, 0, N, N).data;
    saveState();
    for (let i = 0; i < N * N; i++) {
      spatial[i] = 0.299 * d[4 * i] + 0.587 * d[4 * i + 1] + 0.114 * d[4 * i + 2];
    }
    renderSpatial();
    updateFourierFromSpatial();
    URL.revokeObjectURL(u);
  };
  img.src = u;
}

document.getElementById("file").onchange = e => loadImage(e.target.files[0]);
document.getElementById("size").oninput = e => brush = +e.target.value;
document.getElementById("pencil").onclick = () => erase = false;
document.getElementById("eraser").onclick = () => erase = true;

undoBtn.onclick = () => {
  if (historyStack.length > 0) {
    const prevState = historyStack.pop();
    spatial.set(prevState.spatial);
    re.set(prevState.re);
    im.set(prevState.im);
    renderSpatial();
    renderFourier();
    updateUndoButton();
  }
};

document.getElementById("clear").onclick = () => {
  saveState();
  spatial.fill(0);
  re.fill(0);
  im.fill(0);
  renderSpatial();
  renderFourier();
};

document.getElementById("save").onclick = () => {
  let a = document.createElement("a");
  a.download = "resultado_espacial.png";
  a.href = S.toDataURL("image/png");
  a.click();
};

updateUndoButton();

})();
</script>
"""))